In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 2000

df = pd.DataFrame({
    "date": np.random.choice(pd.date_range("2026-07-01", "2026-09-30"), n),
    "fuel_type": np.random.choice(["Petrol", "Diesel"], n, p=[0.75, 0.25]),
    "liters": np.round(np.random.uniform(2, 40, n), 1),
    "payment_method": np.random.choice(["Cash", "Online"], n, p=[0.6, 0.4]),
    "vehicle_type": np.random.choice(["Bike", "Car", "Truck"], n, p=[0.55, 0.38, 0.07]),
})
df["price_per_liter"] = np.where(df["fuel_type"] == "Petrol", 270, 280)
df["amount"] = df["liters"] * df["price_per_liter"]

df.to_csv("petrol_raw.csv", index=False)
print(df.head())
print(df.shape)

        date fuel_type  liters payment_method vehicle_type  price_per_liter  \
0 2026-08-21    Petrol    40.0         Online          Car              270   
1 2026-07-15    Petrol    14.4           Cash         Bike              270   
2 2026-09-10    Petrol    30.4           Cash         Bike              270   
3 2026-08-30    Petrol    32.7         Online         Bike              270   
4 2026-07-21    Diesel    34.6         Online          Car              280   

    amount  
0  10800.0  
1   3888.0  
2   8208.0  
3   8829.0  
4   9688.0  
(2000, 7)


In [2]:
# Raw data ko "gandha" banate hain (asli duniya jaisa)
dirty = df.copy()

# 1. Khali values (kuch entries mein payment ya vehicle likhna bhool gaye)
dirty.loc[dirty.sample(40, random_state=1).index, "payment_method"] = None
dirty.loc[dirty.sample(30, random_state=2).index, "vehicle_type"] = None

# 2. Galat entries (liters negative ya bohat bade)
dirty.loc[dirty.sample(10, random_state=3).index, "liters"] = -5
dirty.loc[dirty.sample(5, random_state=4).index, "liters"] = 500

# 3. Duplicate rows
dirty = pd.concat([dirty, dirty.sample(25, random_state=5)], ignore_index=True)

dirty.to_csv("petrol_raw_dirty.csv", index=False)
print(dirty.shape)
print(dirty.isnull().sum())
print(dirty.duplicated().sum())
print(dirty["liters"].describe())

(2025, 7)
date                0
fuel_type           0
liters              0
payment_method     41
vehicle_type       31
price_per_liter     0
amount              0
dtype: int64
36
count    2025.000000
mean       21.823654
std        26.225275
min        -5.000000
25%        11.200000
50%        20.400000
75%        30.200000
max       500.000000
Name: liters, dtype: float64


In [3]:
clean = dirty.copy()
rows_start = len(clean)

clean = clean.drop_duplicates()
after_dup = len(clean)

clean = clean[(clean["liters"] > 0) & (clean["liters"] <= 100)]
after_liters = len(clean)

clean["payment_method"] = clean["payment_method"].fillna("Unknown")
clean["vehicle_type"] = clean["vehicle_type"].fillna("Unknown")

clean["amount"] = clean["liters"] * clean["price_per_liter"]
clean["date"] = pd.to_datetime(clean["date"])

print("Shuru:", rows_start)
print("Duplicate hatane ke baad:", after_dup)
print("Galat liters hatane ke baad:", after_liters)
print(clean.isnull().sum())

Shuru: 2025
Duplicate hatane ke baad: 1989
Galat liters hatane ke baad: 1974
date               0
fuel_type          0
liters             0
payment_method     0
vehicle_type       0
price_per_liter    0
amount             0
dtype: int64


In [4]:
assert clean.duplicated().sum() == 0, "Duplicates bache hain"
assert clean["liters"].between(0.1, 100).all(), "Liters galat hain"
assert clean.isnull().sum().sum() == 0, "Khali values bachi hain"
print("Saare quality checks pass ho gaye ✅")

Saare quality checks pass ho gaye ✅


In [5]:
clean.to_csv("petrol_clean.csv", index=False)

In [6]:
daily = clean.groupby("date")["amount"].sum().sort_values(ascending=False)
print(daily.head(5))
print("Roz ki average sale:", round(daily.mean(), 2))

date
2026-08-02    208147.0
2026-08-31    203786.0
2026-07-17    192530.0
2026-08-20    188452.0
2026-09-30    187835.0
Name: amount, dtype: float64
Roz ki average sale: 121646.9


In [7]:
clean["month"] = clean["date"].dt.to_period("M")
print(clean.groupby("month")["amount"].sum())

month
2026-07    3857973.0
2026-08    3992182.0
2026-09    3341360.0
Freq: M, Name: amount, dtype: float64


In [8]:
print(clean.groupby("payment_method")["amount"].agg(["count", "sum"]))

                count        sum
payment_method                  
Cash             1183  6673922.0
Online            751  4323287.0
Unknown            40   194306.0


In [9]:
print(clean.groupby("vehicle_type")["amount"].agg(["count", "sum"]))

              count        sum
vehicle_type                  
Bike           1114  6332392.0
Car             711  3982035.0
Truck           119   695812.0
Unknown          30   181276.0
